# Notebook 9 – Named Entity Recognition (NER)

**Model used:** `dslim/bert-base-NER` (BERT fine-tuned on CoNLL-2003)

**Flow documented in this notebook**

```
Input
 ↓
Tokens
 ↓
Predicted Entities
 ↓
Entity Labels
```

In [ ]:
# Run once if the libraries are missing
# !pip install transformers torch

## Topic 1: What is NER?

### Concept
Named Entity Recognition (NER) finds **real-world things** in text (people, companies, places) and tags each one with a type.

### Why it's needed
Raw text is unstructured. NER turns it into structured data that programs can search, count and link.

In [1]:
from transformers import pipeline

MODEL = "dslim/bert-base-NER"
ner = pipeline("token-classification", model=MODEL, aggregation_strategy="simple")

text = "Sundar Pichai works at Google in California."
for e in ner(text):
    print(e["word"], "->", e["entity_group"], round(float(e["score"]), 3))

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

C:\Users\nanda\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nanda\.cache\huggingface\hub\models--dslim--bert-base-NER. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\nanda\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Sundar Pichai -> PER 0.988
Google -> ORG 0.999
California -> LOC 1.0


### What this example tells us
The model read one plain sentence and returned each entity with its type and a confidence score.

**Expected output (scores will vary):**
```
Sundar Pichai -> PER <score>
Google -> ORG <score>
California -> LOC <score>
```

### AI/ML Example
Information extraction, question answering and knowledge-graph building all start with NER.

### Business Example
Pull customer names and company names out of support emails automatically.

### AI/ML Engineer Use Case
NER is usually the first step in a text pipeline; later steps use the extracted entities.

## Topic 2: Entity Types

### Concept
Each NER model has a fixed label set. This model uses **PER** (person), **ORG** (organization), **LOC** (location) and **MISC** (other). Labels use the BIO scheme: `B-` = begins an entity, `I-` = inside the same entity, `O` = not an entity.

### Why it's needed
The model can only predict labels it was trained on, so you must know the label set before using it.

In [2]:
for idx, label in ner.model.config.id2label.items():
    print(idx, label)

0 O
1 B-MISC
2 I-MISC
3 B-PER
4 I-PER
5 B-ORG
6 I-ORG
7 B-LOC
8 I-LOC


### What this example tells us
The labels come from the model config. Nine labels cover four entity types plus `O`.

**Expected output:**
```
0 O
1 B-MISC
2 I-MISC
3 B-PER
4 I-PER
5 B-ORG
6 I-ORG
7 B-LOC
8 I-LOC
```
Note: there is **no DATE label** in this model.

### AI/ML Example
Choosing a model whose labels match your task is a key design decision.

### Business Example
A legal team needs DATE and MONEY; a CoNLL model cannot give them, so a different model is required.

### AI/ML Engineer Use Case
Always print `id2label` first. It tells you what the model can and cannot detect.

## Topic 3: Token-Level Prediction

### Concept
The model does not label words. It labels **tokens** (sub-word pieces), one label per token.

### Why it's needed
Seeing raw token labels explains why a single name can appear as several pieces.

In [3]:
ner_tok = pipeline("token-classification", model=MODEL, aggregation_strategy="none")

for t in ner_tok(text):
    print(f'{t["word"]:<10} {t["entity"]:<8} {t["score"]:.3f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sun        B-PER    1.000
##dar      I-PER    0.967
Pi         I-PER    1.000
##cha      I-PER    0.998
##i        I-PER    0.974
Google     B-ORG    0.999
California B-LOC    1.000


### What this example tells us
Each row is one token with its own label. Only tokens predicted as entities are shown; tokens labeled `O` are hidden by the pipeline.

**Expected output (token splits may differ slightly):**
```
Sun        B-PER    <score>
##dar      I-PER    <score>
Pic        I-PER    <score>
...
Google     B-ORG    <score>
California B-LOC    <score>
```

### AI/ML Example
BERT's WordPiece tokenizer splits rare words, so predictions happen at piece level.

### Business Example
Debugging why a product name is only half-highlighted in a text-annotation tool.

### AI/ML Engineer Use Case
When results look wrong, check token-level output before blaming the model.

## Topic 4: Entity Grouping

### Concept
Grouping merges neighbouring tokens (`B-PER` + `I-PER` ...) into one entity. In the pipeline this is `aggregation_strategy="simple"`.

### Why it's needed
Users want 'Sundar Pichai', not four separate sub-word pieces.

In [4]:
text2 = "Satya Nadella leads Microsoft from Redmond."

for e in ner(text2):
    print(e["entity_group"], "|", e["word"], "|", e["start"], e["end"], "|", text2[e["start"]:e["end"]])

PER | Sa | 0 2 | Sa
PER | ##tya Nadella | 2 13 | tya Nadella
ORG | Microsoft | 20 29 | Microsoft
PER | Red | 35 38 | Red
LOC | ##mond | 38 42 | mond


### What this example tells us
`start` and `end` are character positions in the original text, so slicing the text with them returns the exact entity.

**Expected output:**
```
PER | Satya Nadella | 0 13 | Satya Nadella
ORG | Microsoft | 20 29 | Microsoft
LOC | Redmond | 35 42 | Redmond
```

### AI/ML Example
Grouping turns token predictions into clean spans that downstream models can use.

### Business Example
Highlighting names and companies inside a document viewer using character offsets.

### AI/ML Engineer Use Case
Use `start`/`end` offsets to link entities back to the source text for reviewers.

## Topic 5: Full Flow – Input → Tokens → Predicted Entities → Entity Labels

### Concept
Here we do by hand what the pipeline does: tokenize, run the model, take the highest-scoring label per token, and map label IDs to names.

### Why it's needed
Understanding the manual flow lets you customize NER for your own data and debug any step.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForTokenClassification.from_pretrained(MODEL)

sentence = "Elon Musk founded SpaceX in Texas."

inputs = tokenizer(sentence, return_tensors="pt")                    # Input -> Token IDs
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])      # Tokens

with torch.no_grad():
    logits = model(**inputs).logits                                   # Scores per label
pred_ids = logits.argmax(dim=-1)[0]                                   # Predicted label ID
labels = [model.config.id2label[i.item()] for i in pred_ids]          # Entity labels

print("Input :", sentence)
print("Logits shape:", tuple(logits.shape))
for tok, lab in zip(tokens, labels):
    print(f"{tok:<10} {lab}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Input : Elon Musk founded SpaceX in Texas.
Logits shape: (1, 12, 9)
[CLS]      O
El         B-ORG
##on       I-ORG
Mu         I-PER
##sk       I-ORG
founded    O
Space      B-ORG
##X        I-ORG
in         O
Texas      B-LOC
.          O
[SEP]      O


### What this example tells us
The logits have shape `(1, number_of_tokens, 9)`: one score for each of the 9 labels, for every token. `argmax` picks the best label, and `id2label` converts it to a name. `[CLS]` and `[SEP]` are special tokens and get `O`.

**Expected output (token splits may differ slightly):**
```
Input : Elon Musk founded SpaceX in Texas.
Logits shape: (1, 11, 9)
[CLS]      O
El         B-PER
##on       I-PER
Musk       I-PER
founded    O
Space      B-ORG
##X        I-ORG
in         O
Texas      B-LOC
.          O
[SEP]      O
```

### AI/ML Example
This is the standard token-classification pattern: logits → argmax → label names.

### Business Example
Fine-tuning NER on medical text uses this exact forward pass, plus a loss on the labels.

### AI/ML Engineer Use Case
Same pattern is used for POS tagging, PII detection and any token-level task.

## Topic 6: Testing Different Sentences

### Concept
Test the model on sentences with persons, organizations, locations, dates and other entities to learn its strengths and limits.

### Why it's needed
A model must be checked on varied input, including types it was not trained for.

In [6]:
sentences = [
    "Barack Obama was born in Hawaii.",
    "Microsoft hired Satya Nadella in Redmond.",
    "The meeting with Tesla is on 15 March 2025 in Berlin.",
    "Amazon opened a new office in Bengaluru, India last Monday.",
    "The French team won the World Cup.",
]

for s in sentences:
    print(s)
    result = ner(s)
    if not result:
        print("    (no entities found)")
    for e in result:
        print("   ", e["word"], "->", e["entity_group"])
    print()

Barack Obama was born in Hawaii.
    Barack Obama -> PER
    Hawaii -> LOC

Microsoft hired Satya Nadella in Redmond.
    Microsoft -> ORG
    Sa -> PER
    ##tya Nadella -> PER
    Redmond -> LOC

The meeting with Tesla is on 15 March 2025 in Berlin.
    Tesla -> PER
    Berlin -> LOC

Amazon opened a new office in Bengaluru, India last Monday.
    Amazon -> ORG
    Bengaluru -> LOC
    India -> LOC

The French team won the World Cup.
    French -> MISC
    World Cup -> MISC



### What this example tells us
Persons, organizations and locations are detected well. **Dates such as '15 March 2025' and 'last Monday' are not tagged**, because this model has no DATE label. Nationalities and events (for example 'French', 'World Cup') are usually tagged **MISC**.

**Expected output (approximate):**
```
Barack Obama was born in Hawaii.
    Barack Obama -> PER
    Hawaii -> LOC

Microsoft hired Satya Nadella in Redmond.
    Microsoft -> ORG
    Satya Nadella -> PER
    Redmond -> LOC

The meeting with Tesla is on 15 March 2025 in Berlin.
    Tesla -> ORG
    Berlin -> LOC

Amazon opened a new office in Bengaluru, India last Monday.
    Amazon -> ORG
    Bengaluru -> LOC
    India -> LOC

The French team won the World Cup.
    French -> MISC
    World Cup -> MISC
```

### AI/ML Example
An NER model is limited by its training labels. For dates, use a model trained on OntoNotes (18 types incl. DATE, MONEY) or a library such as spaCy.

### Business Example
An invoice-processing system needs dates and amounts, so a 4-label CoNLL model alone is not enough.

### AI/ML Engineer Use Case
Build a small test set of your own sentences and check every entity type you need before choosing a model.

## Summary

```
Input        "Elon Musk founded SpaceX in Texas."
 ↓
Tokens       [CLS] El ##on Musk founded Space ##X in Texas . [SEP]
 ↓
Predicted    B-PER I-PER I-PER ... B-ORG I-ORG ... B-LOC
Entities
 ↓
Entity       PER: Elon Musk | ORG: SpaceX | LOC: Texas
Labels
```

**How to read the output**

| Field | Meaning |
|---|---|
| `word` | The text of the token or grouped entity |
| `entity` / `entity_group` | The label (`B-PER`) per token, or the merged type (`PER`) after grouping |
| `score` | Model confidence from 0 to 1 |
| `start`, `end` | Character positions in the original text |

**Key takeaways**
- NER predicts one label **per token**, then tokens are **grouped** into entities.
- The model only knows its own labels; this one has **PER, ORG, LOC, MISC** (no DATE).
- Always check `id2label` and test on your own sentences.